In [ ]:
import csv
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Class order is fixed so output ids stay stable
SHAPE_CLASSES = ['circle', 'square', 'triangle', 'none']
COLOR_CLASSES = ['red', 'green', 'blue', 'none']
SHAPE_TO_ID = {k: i for i, k in enumerate(SHAPE_CLASSES)}
COLOR_TO_ID = {k: i for i, k in enumerate(COLOR_CLASSES)}

# Load cell images
cells = np.load('train.tiny.cells.npy').astype(np.float32)
if cells.max() > 1.0:
    cells = cells / 255.0

# Load labels from TSV
shape_labels = []
color_labels = []
with open('train.tiny.cells_labels.txt', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        shape_labels.append(row['shape'])
        color_labels.append(row['color'])

assert len(cells) == len(shape_labels), 'Image/label count mismatch'
shape_ids = np.array([SHAPE_TO_ID[s] for s in shape_labels], dtype=np.int64)
color_ids = np.array([COLOR_TO_ID[c] for c in color_labels], dtype=np.int64)

print('Cells:', cells.shape)
print('Num labels:', len(shape_ids))

class CellDataset(Dataset):
    def __init__(self, x, y_shape, y_color):
        self.x = torch.from_numpy(x).permute(0, 3, 1, 2)
        self.y_shape = torch.from_numpy(y_shape)
        self.y_color = torch.from_numpy(y_color)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y_shape[idx], self.y_color[idx]

class CellCNN(nn.Module):
    def __init__(self, n_shape=4, n_color=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout2d(0.1),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.shape_head = nn.Linear(128, n_shape)
        self.color_head = nn.Linear(128, n_color)

    def forward(self, x):
        h = self.features(x)
        h = self.shared(h)
        return self.shape_head(h), self.color_head(h)

# Train/val split
n = len(cells)
perm = np.random.permutation(n)
split = int(0.9 * n)
train_idx, val_idx = perm[:split], perm[split:]

train_ds = CellDataset(cells[train_idx], shape_ids[train_idx], color_ids[train_idx])
val_ds = CellDataset(cells[val_idx], shape_ids[val_idx], color_ids[val_idx])

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CellCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def evaluate(loader):
    model.eval()
    shape_ok = 0
    color_ok = 0
    both_ok = 0
    total = 0
    with torch.no_grad():
        for xb, ys, yc in loader:
            xb, ys, yc = xb.to(device), ys.to(device), yc.to(device)
            shape_logits, color_logits = model(xb)
            ps = shape_logits.argmax(dim=1)
            pc = color_logits.argmax(dim=1)
            shape_ok += (ps == ys).sum().item()
            color_ok += (pc == yc).sum().item()
            both_ok += ((ps == ys) & (pc == yc)).sum().item()
            total += xb.size(0)
    return shape_ok / total, color_ok / total, both_ok / total

EPOCHS = 100
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for xb, ys, yc in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False):
        xb, ys, yc = xb.to(device), ys.to(device), yc.to(device)
        shape_logits, color_logits = model(xb)
        loss = criterion(shape_logits, ys) + criterion(color_logits, yc)
        opt.zero_grad()
        loss.backward()
        opt.step()
        running_loss += loss.item() * xb.size(0)

    train_shape_acc, train_color_acc, train_both_acc = evaluate(train_loader)
    val_shape_acc, val_color_acc, val_both_acc = evaluate(val_loader)
    print(
        f'Epoch {epoch:02d} | loss={running_loss/len(train_ds):.4f} | '
        f'train_shape={train_shape_acc:.4f} train_color={train_color_acc:.4f} train_both={train_both_acc:.4f} | '
        f'val_shape={val_shape_acc:.4f} val_color={val_color_acc:.4f} val_both={val_both_acc:.4f}'
    )

# Quick sample predictions
model.eval()
sample_x = torch.from_numpy(cells[:16]).permute(0, 3, 1, 2).to(device)
with torch.no_grad():
    s_log, c_log = model(sample_x)
pred_s = s_log.argmax(dim=1).cpu().numpy()
pred_c = c_log.argmax(dim=1).cpu().numpy()

print('\nSample predictions (first 16):')
for i in range(16):
    print(i, SHAPE_CLASSES[pred_s[i]], COLOR_CLASSES[pred_c[i]])

torch.save(
    {
        'model_state_dict': model.state_dict(),
        'shape_classes': SHAPE_CLASSES,
        'color_classes': COLOR_CLASSES,
    },
    'cell_cnn_shape_color.pt'
)
print('\nSaved model to cell_cnn_shape_color.pt')